# 01 — Encode images with BiomedCLIP + zero-shot (Phase 4)

Runs the repo's `src/embed.py` and `src/zeroshot.py` on a GPU. All logic lives in the repo;
this notebook only sets up Colab, runs it, and saves everything to Google Drive.

**Before the first run (one-time):**
1. `Runtime → Change runtime type → T4 GPU`
2. Google Drive contains `MyDrive/blastocyst-clip/data/` with `blastocyst/` and `captions/`
   (upload the local `data/` folder's contents there)
3. Colab Secrets (key icon, left sidebar): add `GITHUB_TOKEN` = your GitHub fine-grained token,
   and switch on *Notebook access*

Outputs go to `MyDrive/blastocyst-clip/results/`, so they survive the session ending.

In [ ]:
# 1. GPU check — should show a Tesla T4. If not: Runtime → Change runtime type.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/blastocyst-clip'
import os
for p in ['data/blastocyst/Images', 'data/captions/images_2']:
    full = os.path.join(DRIVE, p)
    print(p, '->', len(os.listdir(full)) if os.path.isdir(full) else 'MISSING')

In [ ]:
# 3. Get the code (private repo): clone the first time, pull afterwards
from google.colab import userdata
REPO = '/content/blastocyst-clip'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/Daniel0813/blastocyst-clip.git'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q {url} {REPO}
del token, url
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Install the one missing dependency and record versions
!pip install -q open_clip_torch
import torch, open_clip, timm, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| open_clip', open_clip.__version__,
      '| timm', timm.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 5. Label table and caption subset (rebuilt from the raw files if they are not on Drive yet)
if not os.path.exists(f'{DRIVE}/data/blastocyst/labels.csv'):
    !python src/build_labels.py --data-dir {DRIVE}/data/blastocyst
if not os.path.exists(f'{DRIVE}/data/captions/blastocyst_captions.jsonl'):
    !python src/select_blastocyst_captions.py --captions-dir {DRIVE}/data/captions

In [ ]:
# 6. Encode all 2,342 blastocyst images, the 172 caption images, and the Gardner prompts
!python -W ignore src/embed.py --data-dir {DRIVE}/data --model biomedclip --out-dir {DRIVE}/results/embeddings/biomedclip

In [ ]:
# 7. E0: zero-shot grading on the 300-image test set
!python -W ignore src/zeroshot.py --emb-dir {DRIVE}/results/embeddings/biomedclip --labels {DRIVE}/data/blastocyst/labels.csv

In [ ]:
# 8. Record which code produced these results
import json, subprocess, datetime
info = {'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip(),
        'date': datetime.datetime.now().isoformat(timespec='seconds'),
        'torch': torch.__version__, 'open_clip': open_clip.__version__,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
json.dump(info, open(f'{DRIVE}/results/embeddings/biomedclip/run_info.json', 'w'), indent=2)
print(info)